In [6]:
import warnings
warnings.filterwarnings("ignore")

In [16]:
import ast
import pandas as pd
from tqdm import tqdm

In [8]:
metadata = pd.read_csv('./APS Data/metadata.csv')

### Author Table

In [9]:
necessary_columns = metadata[['id','authors','date','affiliations']]

In [10]:
necessary_columns.to_csv('./Created Data Tables/necessary_columns.csv', index=False)

In [13]:
necessary_columns = pd.read_csv('./Created Data Tables/necessary_columns.csv')

In [14]:
# set the max_colwidth option to None to display full contents of each column
pd.set_option('display.max_colwidth', None)

selected_row = necessary_columns.iloc[1] 
print(selected_row)

id                                                                                                                    10.1103/PhysRev.1.124
authors         [{'type': 'Person', 'name': 'Lachlan Gilchrist', 'firstname': 'Lachlan', 'surname': 'Gilchrist', 'affiliationIds': ['a1']}]
date                                                                                                                             1913-02-01
affiliations                                                   [{'id': 'a1', 'name': 'Ryerson Physical Laboratory, University of Chicago'}]
Name: 1, dtype: object


In [17]:
# function to parse string representation to list of dictionaries
def parse_authors(authors_str):
    try:
        return ast.literal_eval(authors_str)
    except (SyntaxError, ValueError):
        return []

# function to parse string representation to list of dictionaries for affiliations
def parse_affiliations(affiliations_str):
    try:
        return ast.literal_eval(affiliations_str)
    except (SyntaxError, ValueError):
        return []

# function to extract information for each author and store it as a dictionary
def extract_authors_info(row, affiliations_dict):
    authors_info = []
    date_str = row.get('date', None)
    publication_year = date_str[:4] if date_str and pd.notna(date_str) else None
    for author in row['authors']:
        # get affiliation names from affiliation IDs
        affiliation_ids = author.get('affiliationIds', [])
        affiliation_names = [affiliations_dict.get(aff_id) for aff_id in affiliation_ids if aff_id in affiliations_dict]
        affiliation_name = ', '.join(affiliation_names) if affiliation_names else None

        author_info = {
            'doi': row['id'],
            'type': author.get('type'),
            'name': author.get('name'),
            'firstname': author.get('firstname'),
            'surname': author.get('surname'),
            'year': publication_year,
            'affiliation': affiliation_name
        }
        authors_info.append(author_info)
    return authors_info

# read the CSV file in chunks
chunksize = 10000  
authors_info_chunks = []

# initialize an empty dictionary for affiliations
affiliations_dict = {}

# read the CSV file to build the affiliations dictionary
for chunk in tqdm(pd.read_csv('./Created Data Tables/necessary_columns.csv', chunksize=chunksize), desc="Building Affiliations Dictionary"):
    chunk['affiliations'].fillna('[]', inplace=True)
    chunk['affiliations'] = chunk['affiliations'].apply(parse_affiliations)
    
    for _, row in chunk.iterrows():
        for affiliation in row['affiliations']:
            affiliations_dict[affiliation['id']] = affiliation['name']

# process the CSV file to extract authors information
for chunk in tqdm(pd.read_csv('./Created Data Tables/necessary_columns.csv', chunksize=chunksize), desc="Extracting Authors Information"):
    chunk['authors'].fillna('[]', inplace=True)
    chunk['authors'] = chunk['authors'].apply(parse_authors)
    chunk['authors_info'] = chunk.apply(extract_authors_info, axis=1, affiliations_dict=affiliations_dict)
    authors_info_chunks.append(chunk.explode('authors_info')['authors_info'].apply(pd.Series))

# concatenate the processed chunks into a single dataframe
authors_info = pd.concat(authors_info_chunks)

# remove rows with all NaN values
authors_info_cleaned = authors_info.dropna(how='all')

# select only the necessary columns
authors_info_cleaned = authors_info_cleaned[['doi', 'type', 'name', 'firstname', 'surname', 'year', 'affiliation']]

# drop duplicate entries
authors_info_cleaned = authors_info_cleaned.drop_duplicates()

# reset index
authors_info_cleaned.reset_index(drop=True, inplace=True)

# create the identifier column
authors_info_cleaned['identifier'] = authors_info_cleaned.index

# move the 'identifier' column to the first position
columns = authors_info_cleaned.columns.tolist()
columns = ['identifier'] + [col for col in columns if col != 'identifier']
authors_info_cleaned = authors_info_cleaned[columns]

Building Affiliations Dictionary: 73it [02:17,  1.89s/it]
Extracting Authors Information: 73it [16:38, 13.68s/it]


In [18]:
# list of countries
countries = [
    "Afghanistan", "Albania", "Algeria", "Andorra", "Angola", "Antigua and Barbuda", "Argentina", "Armenia", "Australia", "Austria",
    "Azerbaijan", "Bahamas", "Bahrain", "Bangladesh", "Barbados", "Belarus", "Belgium", "Belize", "Benin", "Bhutan", "Bolivia", 
    "Bosnia and Herzegovina", "Botswana", "Brazil", "Brunei", "Bulgaria", "Burkina Faso", "Burundi", "Cabo Verde", "Cambodia", "Cameroon",
    "Canada", "Central African Republic", "Chad", "Chile", "China", "Colombia", "Comoros", "Congo, Democratic Republic of the", 
    "Congo, Republic of the", "Costa Rica", "Croatia", "Cuba", "Cyprus", "Czech Republic", "Denmark", "Djibouti", "Dominica", "Dominican Republic",
    "East Timor", "Ecuador", "Egypt", "El Salvador", "Equatorial Guinea", "Eritrea", "Estonia", "Eswatini", "Ethiopia", "Fiji", "Finland", 
    "France", "Gabon", "Gambia", "Georgia", "Germany", "Ghana", "Greece", "Grenada", "Guatemala", "Guinea", "Guinea-Bissau", "Guyana", "Haiti",
    "Honduras", "Hungary", "Iceland", "India", "Indonesia", "Iran", "Iraq", "Ireland", "Israel", "Italy", "Jamaica", "Japan", "Jordan",
    "Kazakhstan", "Kenya", "Kiribati", "Korea, North", "Korea, South", "Kosovo", "Kuwait", "Kyrgyzstan", "Laos", "Latvia", "Lebanon", "Lesotho",
    "Liberia", "Libya", "Liechtenstein", "Lithuania", "Luxembourg", "Madagascar", "Malawi", "Malaysia", "Maldives", "Mali", "Malta", "Marshall Islands",
    "Mauritania", "Mauritius", "Mexico", "Micronesia", "Moldova", "Monaco", "Mongolia", "Montenegro", "Morocco", "Mozambique", "Myanmar", "Namibia",
    "Nauru", "Nepal", "Netherlands", "New Zealand", "Nicaragua", "Niger", "Nigeria", "North Macedonia", "Norway", "Oman", "Pakistan", "Palau",
    "Panama", "Papua New Guinea", "Paraguay", "Peru", "Philippines", "Poland", "Portugal", "Qatar", "Romania", "Russia", "Rwanda", "Saint Kitts and Nevis",
    "Saint Lucia", "Saint Vincent and the Grenadines", "Samoa", "San Marino", "Sao Tome and Principe", "Saudi Arabia", "Senegal", "Serbia", "Seychelles",
    "Sierra Leone", "Singapore", "Slovakia", "Slovenia", "Solomon Islands", "Somalia", "South Africa", "South Sudan", "Spain", "Sri Lanka", "Sudan",
    "Suriname", "Sweden", "Switzerland", "Syria", "Taiwan", "Tajikistan", "Tanzania", "Thailand", "Togo", "Tonga", "Trinidad and Tobago", "Tunisia",
    "Turkey", "Turkmenistan", "Tuvalu", "Uganda", "Ukraine", "United Arab Emirates", "United Kingdom", "United States", "Uruguay", "Uzbekistan",
    "Vanuatu", "Vatican City", "Venezuela", "Vietnam", "Yemen", "Zambia", "Zimbabwe"
]

# function to extract countries from affiliation
def extract_countries(affiliation):
    if pd.isna(affiliation) or affiliation.strip() == "":
        return []
    found_countries = [country for country in countries if country in affiliation]
    return found_countries if found_countries else []

# apply the function to create a new column 'countries'
tqdm.pandas(desc="Extracting countries")
authors_info_cleaned['countries'] = authors_info_cleaned['affiliation'].progress_apply(extract_countries)

Extracting countries: 100%|██████████| 2571821/2571821 [03:10<00:00, 13530.58it/s]


In [19]:
authors_info_cleaned.to_csv('./Created Data Tables/author_table.csv', index=False)

In [20]:
author_table = pd.read_csv('./Created Data Tables/author_table.csv')

In [21]:
print(author_table)

         identifier                           doi    type  \
0                 0         10.1103/PhysRev.1.124  Person   
1                 1         10.1103/PhysRev.1.141  Person   
2                 2         10.1103/PhysRev.1.154  Person   
3                 3          10.1103/PhysRev.1.16  Person   
4                 4         10.1103/PhysRev.1.161  Person   
...             ...                           ...     ...   
2571816     2571816  10.1103/RevModPhys.94.045008  Person   
2571817     2571817  10.1103/RevModPhys.94.045008  Person   
2571818     2571818  10.1103/RevModPhys.94.045008  Person   
2571819     2571819  10.1103/RevModPhys.94.045008  Person   
2571820     2571820  10.1103/RevModPhys.94.045008  Person   

                             name       firstname        surname  year  \
0               Lachlan Gilchrist         Lachlan      Gilchrist  1913   
1                E. E. Somermeier           E. E.     Somermeier  1913   
2               Chester A. Butman      Cheste

In [22]:
# earliest and latest year
earliest_year = author_table['year'].min()
latest_year = author_table['year'].max()
print(f"Earliest year: {earliest_year}")
print(f"Latest year: {latest_year}")
print()

# most common years
most_common_years = author_table['year'].value_counts().head(10)
print("Most common years:")
print(most_common_years)
print()

# distribution of the number of entries per year
year_distribution = author_table['year'].value_counts()
print("Distribution of the number of entries per year:")
print(year_distribution.describe())
print()

# count the number of unique affiliations
unique_affiliations = author_table['affiliation'].nunique()
print(f"Number of unique affiliations: {unique_affiliations}")
print()

# most common affiliations
most_common_affiliations = author_table['affiliation'].value_counts().head(10)
print("Most common affiliations:")
print(most_common_affiliations)
print()

# distribution of the number of entries per affiliation
affiliation_distribution = author_table['affiliation'].value_counts()
print("Distribution of the number of entries per affiliation:")
print(affiliation_distribution.describe())
print()

# count the number of unique countries 
unique_countries = author_table['countries'].explode().nunique()
print(f"Number of unique countries: {unique_countries}")
print()

# most common countries 
most_common_countries = author_table['countries'].explode().value_counts().head(10)
print("Most common countries:")
print(most_common_countries)
print()

# distribution of the number of entries per country 
country_distribution = author_table['countries'].explode().value_counts()
print("Distribution of the number of entries per country:")
print(country_distribution.describe())
print()

# count empty countries
empty_country_count = author_table[author_table['countries'].apply(len) == 0].shape[0]
print(f"Number of empty country columns: {empty_country_count}")

Earliest year: 1893
Latest year: 2022

Most common years:
year
2020    99721
2021    93824
2022    91457
2019    90296
2018    86671
2017    83867
2012    82977
2011    79801
2016    79693
2013    78011
Name: count, dtype: int64

Distribution of the number of entries per year:
count      130.000000
mean     19783.238462
std      28586.375727
min         20.000000
25%        217.000000
50%       3307.000000
75%      29502.000000
max      99721.000000
Name: count, dtype: float64

Number of unique affiliations: 1292

Most common affiliations:
affiliation
Centro de Física de Materiales CSIC/UPV-EHU-Materials Physics Center, Manuel Lardizabal 5, E-20018 San Sebastián, Spain, Donostia International Physics Center, Paseo Manuel Lardizabal 4, E-20018 Donostia-San Sebastián, Spain, and Physics Department E20, Technical University of Munich, 85748 Garching, Germany                                                                                                                                     

In [23]:
# filter the df to include only instances where the 'type' is equal to "Person"
person_types = author_table[author_table['type'] == 'Person']

# print how many times this occurs in the dataset
print("Number of instances where 'type' is equal to 'Person':", len(person_types))
print()

# print the instances where the 'type' is equal to "Person"
print("Instances where 'type' is equal to 'Person':")
print(person_types)

Number of instances where 'type' is equal to 'Person': 2570491

Instances where 'type' is equal to 'Person':
         identifier                           doi    type  \
0                 0         10.1103/PhysRev.1.124  Person   
1                 1         10.1103/PhysRev.1.141  Person   
2                 2         10.1103/PhysRev.1.154  Person   
3                 3          10.1103/PhysRev.1.16  Person   
4                 4         10.1103/PhysRev.1.161  Person   
...             ...                           ...     ...   
2571816     2571816  10.1103/RevModPhys.94.045008  Person   
2571817     2571817  10.1103/RevModPhys.94.045008  Person   
2571818     2571818  10.1103/RevModPhys.94.045008  Person   
2571819     2571819  10.1103/RevModPhys.94.045008  Person   
2571820     2571820  10.1103/RevModPhys.94.045008  Person   

                             name       firstname        surname  year  \
0               Lachlan Gilchrist         Lachlan      Gilchrist  1913   
1         

In [24]:
# filter the df to include only instances where the 'type' is equal to "Collaboration"
collaboration_types = author_table[author_table['type'] == 'Collaboration']

# print how many times this occurs in the dataset
print("Number of instances where 'type' is equal to 'Collaboration':", len(collaboration_types))
print()

# print the instances where the 'type' is equal to "Collaboration"
print("Instances where 'type' is equal to 'Collaboration':")
print(collaboration_types)

Number of instances where 'type' is equal to 'Collaboration': 1330

Instances where 'type' is equal to 'Collaboration':
         identifier                       doi           type  \
5652           5652   10.1103/PhysRev.107.325  Collaboration   
14756         14756  10.1103/PhysRev.122.1286  Collaboration   
29112         29112   10.1103/PhysRev.139.AB3  Collaboration   
33569         33569  10.1103/PhysRev.148.1192  Collaboration   
33968         33968  10.1103/PhysRev.149.1044  Collaboration   
...             ...                       ...            ...   
2567777     2567777  10.1103/RevModPhys.56.S1  Collaboration   
2567846     2567846  10.1103/RevModPhys.57.S1  Collaboration   
2567860     2567860  10.1103/RevModPhys.57.S1  Collaboration   
2567950     2567950  10.1103/RevModPhys.59.S1  Collaboration   
2567968     2567968  10.1103/RevModPhys.59.S1  Collaboration   

                                                                               name  \
5652                    

### Paper Table

In [25]:
# load the dataset
file_path = "./APS Data/aps-dataset-citations-2022.csv"
citing_cited = pd.read_csv(file_path)

# define the paper table
paper_table = pd.DataFrame(columns=['identifier', 'doi'])

# merge and drop duplicates to ensure uniqueness
paper_table['doi'] = pd.concat([citing_cited['citing_doi'], citing_cited['cited_doi']]).drop_duplicates().reset_index(drop=True)

# add identifier column
paper_table['identifier'] = range(0, len(paper_table))

In [26]:
paper_table.to_csv('./Created Data Tables/paper_table.csv', index=False)

In [27]:
paper_table = pd.read_csv('./Created Data Tables/paper_table.csv')

In [28]:
print(paper_table)

        identifier                             doi
0                0   10.1103/PhysRevSeriesI.11.215
1                1   10.1103/PhysRevSeriesI.12.121
2                2     10.1103/PhysRevSeriesI.7.93
3                3   10.1103/PhysRevSeriesI.16.267
4                4    10.1103/PhysRevSeriesI.17.65
...            ...                             ...
709798      709798     10.1103/PhysRevD.105.129903
709799      709799      10.1103/PRXEnergy.1.017001
709800      709800     10.1103/PhysRevE.106.029901
709801      709801     10.1103/PhysRevD.106.069901
709802      709802  10.1103/PhysRevFluids.7.093104

[709803 rows x 2 columns]


### Merged Table

In [29]:
# merge the author_table and paper_table on a common key
merged_table = pd.merge(author_table, paper_table, left_on='doi', right_on='doi')

# rename the identifier columns
merged_table.rename(columns={'identifier_x': 'author_identifier', 'identifier_y': 'paper_identifier'}, inplace=True)

In [30]:
# print the range of author indices
print("Range of author indices:", merged_table['author_identifier'].min(), "-", merged_table['author_identifier'].max())

# print the range of paper indices
print("Range of paper indices:", merged_table['paper_identifier'].min(), "-", merged_table['paper_identifier'].max())

Range of author indices: 0 - 2571820
Range of paper indices: 0 - 709802


In [31]:
# find the maximum index value for authors and papers
max_author_index = merged_table['author_identifier'].max()
max_paper_index = merged_table['paper_identifier'].max()

# add an offset to the indices of papers to avoid overlap with authors
offset = max_author_index + 1

# update the paper indices by adding the offset
merged_table['paper_identifier'] += offset

In [32]:
# print the range of author indices
print("Range of author indices:", merged_table['author_identifier'].min(), "-", merged_table['author_identifier'].max())

# print the range of paper indices
print("Range of paper indices:", merged_table['paper_identifier'].min(), "-", merged_table['paper_identifier'].max())

Range of author indices: 0 - 2571820
Range of paper indices: 2571821 - 3281623


In [33]:
merged_table.to_csv('./Created Data Tables/merged_table.csv', index=False)

In [34]:
merged_table = pd.read_csv('./Created Data Tables/merged_table.csv')

In [35]:
print(merged_table)

         author_identifier                           doi    type  \
0                        0         10.1103/PhysRev.1.124  Person   
1                        3          10.1103/PhysRev.1.16  Person   
2                        7           10.1103/PhysRev.1.2  Person   
3                        9         10.1103/PhysRev.1.218  Person   
4                       12         10.1103/PhysRev.1.259  Person   
...                    ...                           ...     ...   
2528889            2571816  10.1103/RevModPhys.94.045008  Person   
2528890            2571817  10.1103/RevModPhys.94.045008  Person   
2528891            2571818  10.1103/RevModPhys.94.045008  Person   
2528892            2571819  10.1103/RevModPhys.94.045008  Person   
2528893            2571820  10.1103/RevModPhys.94.045008  Person   

                             name       firstname        surname  year  \
0               Lachlan Gilchrist         Lachlan      Gilchrist  1913   
1             David W. Cornelius.  